### Text Gen with LSTM 

Notebook does Text Gen with a simple LSTM model with the following layers: Embedding, LSTM, Dense

Data comes from  https://s3.amazonaws.com/text-datasets/nietzsche.txt

Use as prompt: "new faculty and the jubilation reached its climax when kant" to generate text



In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import re

2025-03-29 00:56:26.605050: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743209786.651319     329 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743209786.664109     329 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-29 00:56:26.762677: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def preprocess_text(text):
    """Cleans and tokenizes the text."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    return tokens



The function 

        create_Sequences(tokens, seq_lengt)
        
takes a list of tokens and an integer seq_length as input to generate a list of sequences from the given tokens.

+ The primary goal is to transform a flat list of tokens into a set of **overlapping sequences** which are commonly used in NLP and/or time-series prediction.

+ Each sequence consists of seq_length input tokens followed by one output token.

+ tokens: is a list of items (e.g., words, characters, numbers). These are the raw data from which the sequences will be constructed.

+ seq_length: An integer that determines the length of the input portion of each sequence.

+ Example: 

+ If tokens = [1, 2, 3, 4, 5, 6] and seq_length is 3, the output is :

    `[[1, 2, 3, 4], [2, 3, 4, 5], [3, 4, 5, 6]]`

Note that each inner list contains 4 elements. The first three are the input, and the last is the output.


In [3]:
##     Create input-output sequences

def create_sequences(tokens, seq_length):
    """Creates input-output sequences."""
    sequences = []
    for i in range(seq_length, len(tokens)):
        seq = tokens[i - seq_length:i + 1]
        sequences.append(seq)
    return sequences


In [4]:
 ##     Create token-to-index and index-to-token mappings

def create_token_index(tokens):
    """Creates token-to-index and index-to-token mappings."""
    unique_tokens = sorted(list(set(tokens)))
    token_index = {token: index for index, token in enumerate(unique_tokens)}
    index_token = {index: token for index, token in enumerate(unique_tokens)}
    return token_index, index_token

In [5]:
##   Create input and output datasets.

def create_dataset(sequences, token_index, seq_length):
    """Creates input and output datasets."""
    x = []
    y = []
    for seq in sequences:
        input_seq = seq[:-1]
        output_token = seq[-1]
        x.append([token_index[token] for token in input_seq])
        y.append(token_index[output_token])
    x = np.array(x)
    y = np.array(y)
    return x, y

In [6]:
## The code in this cell did not work. Ignore

def build_lstm_model_bad(vocab_size, embedding_dim, rnn_units, batch_size):
    """Builds the LSTM model."""
    model = keras.Sequential([
        keras.layers.Embedding(vocab_size, embedding_dim, batch_input_shape=[batch_size, None]), # Correct usage
        keras.layers.LSTM(rnn_units, return_sequences=True, stateful=True, recurrent_initializer='glorot_uniform'),
        keras.layers.Dense(vocab_size)
    ])
    return model

In [7]:

##    Build the LSTM model

def build_lstm_model(vocab_size, embedding_dim, rnn_units, batch_size):
    """Builds the LSTM model."""
    model = keras.Sequential([
        keras.layers.Embedding(vocab_size, embedding_dim), # Removed batch_input_shape
        # keras.layers.LSTM(rnn_units, return_sequences=True, stateful=True, recurrent_initializer='glorot_uniform'),

        # keras.layers.LSTM(rnn_units, return_sequences=True, recurrent_initializer='glorot_uniform'),
        keras.layers.LSTM(rnn_units, recurrent_initializer='glorot_uniform'),

        keras.layers.Dense(vocab_size)
    ])
    return model

In [8]:

##    Generates text using the trained LSTM model. Note temperature=1.0

def generate_text_lstm(model, start_string, token_index, index_token, num_generate, seq_length, temperature=1.0):
    """Generates text using the trained LSTM model."""
    input_eval = [token_index[s] for s in start_string.split()]
    input_eval = tf.expand_dims(input_eval, 0) #Keep this.
    text_generated = start_string.split()

    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(tf.expand_dims(predictions, 0), num_samples=1)[-1, 0].numpy() #Added expand_dims here.
        text_generated.append(index_token[predicted_id])
        input_eval = tf.expand_dims([predicted_id], 0)

    return ' '.join(text_generated)

In [9]:
## Dataset is the collected works by F. Nietzsche

path = keras.utils.get_file(
'nietzsche.txt',
origin='https://s3.amazonaws.com/text-datasets/nietzsche.txt')
text = open(path).read().lower()
print('Corpus length:', len(text))

600901/600901 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Corpus length: 600893


In [ ]:

## Simpler text used for testing. 

# Example usage: Use text from cell above 
# #text = """
#The quick brown fox jumps over the lazy dog.
#The dog was very lazy.
#The fox was very quick.
#"""


In [10]:
tokens = preprocess_text(text)
seq_length = 5
sequences = create_sequences(tokens, seq_length)
token_index, index_token = create_token_index(tokens)
vocab_size = len(token_index)
x, y = create_dataset(sequences, token_index, seq_length)

In [11]:
embedding_dim = 256
rnn_units = 1024
batch_size = 64


In [12]:
lstm_model = build_lstm_model(vocab_size, embedding_dim, rnn_units, batch_size)
lstm_model.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))

I0000 00:00:1743209844.598389     329 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9433 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [ ]:
# Train the LSTM model
lstm_model.fit(x, y, epochs=50, batch_size=batch_size)

#50 epochs 20min


Epoch 1/50


I0000 00:00:1743209848.922281     395 cuda_dnn.cc:529] Loaded cuDNN version 90300


1549/1549 ━━━━━━━━━━━━━━━━━━━━ 27s 16ms/step - loss: 6.9933
Epoch 2/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 6.0276
Epoch 3/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 5.4239
Epoch 4/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 4.6543
Epoch 5/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 3.6042
Epoch 6/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 2.5068
Epoch 7/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 1.5736
Epoch 8/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.8455
Epoch 9/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.3893
Epoch 10/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.1631
Epoch 11/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 24s 16ms/step - loss: 0.0857
Epoch 12/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0753
Epoch 13/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0867
Epoch 14/50
1549/1549 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0702
Epoch 15/50


In [14]:
lstm_model.save("./Playground/TrainedLSTM_E50.keras")

In [15]:
# Generate Text
# start_string = "the quick brown fox"
start_string = "new faculty and the jubilation reached its climax when kant" 
num_generate = 20
generated_text = generate_text_lstm(lstm_model, start_string, token_index, index_token, num_generate, seq_length)
print(generated_text)

new faculty and the jubilation reached its climax when kant minds to a man who is not to moral will to a man with one who is a man with


new faculty and the jubilation reached its climax when kant feel all love of the most strength of the most ever absolutely after all his own love of all nature

In [ ]:
# Generate Text
# start_string = "the quick brown fox"
start_string = "new faculty and the jubilation reached its climax when kant" 
num_generate = 20
generated_text = generate_text_lstm(lstm_model, start_string, token_index, index_token, num_generate, seq_length, 0.5)
print(generated_text)

new faculty and the jubilation reached its climax when kant up to a man who is not to a man who is not to a man is not to a


new faculty and the jubilation reached its climax when kant further god they are thus than to a right here too much god to a right here too much god

In [18]:
# Generate Text
# start_string = "the quick brown fox"
start_string = "new faculty and the jubilation reached its climax when kant" 
num_generate = 100
generated_text = generate_text_lstm(lstm_model, start_string, token_index, index_token, num_generate, seq_length, 0.5)
print(generated_text)

new faculty and the jubilation reached its climax when kant up to a man who is a man who is not to a man who finally to a man who is a man who finally to the first wait for example of the german loves them as a man who so much more powerful attest the more dangerous tragedymay no longer say to the german loves them as a man who is to the german opinion is a man who knows that the german spirit has been more dangerous afford it is to a man has been more dangerous knowledge that the most dangerous affording the most dangerous tragedymay have


In [17]:
# Generate Text
# start_string = "the quick brown fox"
start_string = "new faculty and the jubilation reached its climax when kant" 
num_generate = 20
generated_text = generate_text_lstm(lstm_model, start_string, token_index, index_token, num_generate, seq_length, 5.0)
print(generated_text)

new faculty and the jubilation reached its climax when kant namely derivatively name denote mussulmans sharpsightedness inclines believing understanding around subservient laughand collectors ye selfdenial wherever auditors elaborations where paradox


new faculty and the jubilation reached its climax when kant scourge metaphysics assailant property firstlings reduce gay accepted origin dread tongues rates prey relax shake inapplicable begloom permanent man selfconfidence

